**Import libraries**

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import notebookutils
import uuid
from datetime import datetime

StatementMeta(, 3ccb94be-7837-4c8e-ab11-f1817ece156f, 5, Finished, Available, Finished, False)

**Define constants**

In [4]:
INCOMING_DIR = "Files/Incoming"
BRONZE_ARCHIVE_BASE = "Files/BronzeArchive"

BRONZE_TABLE = "dbo.sales_bronze_raw"
SILVER_TABLE = "dbo.sales_silver3"

NOTEBOOK_NAME = "SilverNotebook"
RUN_ID = str(uuid.uuid4())

# If you want to process only specific files, list them here.
# Leave empty [] to process every CSV found in Incoming.
FILES_TO_PROCESS = []

# Optional reset flags
RESET_BRONZE = False
RESET_SILVER = False


StatementMeta(, 3ccb94be-7837-4c8e-ab11-f1817ece156f, 6, Finished, Available, Finished, False)

**Create Bronze and Silver tables if they do not exist**

In [78]:
DeltaTable.createIfNotExists(spark) \
    .tableName(BRONZE_TABLE) \
    .addColumn("SalesOrderNumber", StringType()) \
    .addColumn("SalesOrderLineNumber", IntegerType()) \
    .addColumn("OrderDate", StringType()) \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("Item", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", DoubleType()) \
    .addColumn("Tax", DoubleType()) \
    .addColumn("SnapshotDate", StringType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("SourceFilePath", StringType()) \
    .addColumn("ArchivePath", StringType()) \
    .addColumn("IngestedAt", TimestampType()) \
    .addColumn("RunId", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(SILVER_TABLE) \
    .addColumn("SalesOrderNumber", StringType()) \
    .addColumn("SalesOrderLineNumber", IntegerType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("Item", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", DoubleType()) \
    .addColumn("Tax", DoubleType()) \
    .addColumn("SnapshotDate", DateType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()

if RESET_BRONZE:
    spark.sql(f"DELETE FROM {BRONZE_TABLE}")

if RESET_SILVER:
    spark.sql(f"DELETE FROM {SILVER_TABLE}")


StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 80, Finished, Available, Finished, False)

**Define raw schema**

In [79]:
sales_schema = StructType([
    StructField("SalesOrderNumber", StringType(), True),
    StructField("SalesOrderLineNumber", IntegerType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("Tax", DoubleType(), True),
    StructField("SnapshotDate", StringType(), True)
])

StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 81, Finished, Available, Finished, False)

**List files in Incoming**

In [80]:
all_items = notebookutils.fs.ls(INCOMING_DIR)
incoming_csv_files = [x for x in all_items if (not getattr(x, "isDir", False)) and x.name.lower().endswith(".csv")]

if FILES_TO_PROCESS:
    incoming_csv_files = [x for x in incoming_csv_files if x.name in FILES_TO_PROCESS]

print("Files to process:")
for f in incoming_csv_files:
    print("-", f.name)


StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 82, Finished, Available, Finished, False)

Files to process:
- Sales_2019_20260404_V1.csv
- Sales_2020_2026_0404_V1.csv
- Sales_2020_2026_0404_V2copy.csv


**Copy each landed file to Bronze archive and load Bronze raw table**

In [81]:
import re

def bronze_file_already_loaded(file_name: str) -> bool:
    return (
        spark.read.table(BRONZE_TABLE)
        .filter(col("SourceFileName") == file_name)
        .limit(1)
        .count() > 0
    )

bronze_batch_df = None

for file_info in incoming_csv_files:
    file_name = file_info.name
    source_path = file_info.path

    # Skip file if it has already been loaded into Bronze
    if bronze_file_already_loaded(file_name):
        print(f"Skipping already loaded Bronze file: {file_name}")
        continue

    # Extract year from file name, e.g. Sales_2020.csv -> 2020
    year_match = re.search(r"(20\d{2})", file_name)

    if year_match:
        year_part = year_match.group(1)
    else:
        raise ValueError(f"Could not find year in file name: {file_name}")

    # Keep only year folder so all versions for the same year stay together
    archive_folder = f"{BRONZE_ARCHIVE_BASE}/year={year_part}"
    archive_path = f"{archive_folder}/{file_name}"

    # Create year folder if needed
    notebookutils.fs.mkdirs(archive_folder)

    # Copy raw file from Incoming to Bronze archive
    notebookutils.fs.cp(source_path, archive_path, True)

    # Read raw file from Incoming shortcut path
    raw_df = (
        spark.read.format("csv")
        .option("header", "true")
        .schema(sales_schema)
        .load(source_path)
    )

    # Add Bronze metadata
    one_file_bronze_df = (
        raw_df
        .withColumn("SourceFileName", lit(file_name))
        .withColumn("SourceFilePath", lit(source_path))
        .withColumn("ArchivePath", lit(archive_path))
        .withColumn("IngestedAt", current_timestamp())
        .withColumn("RunId", lit(RUN_ID))
    )

    # Append raw rows into Bronze raw Delta table
    one_file_bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

    # Keep this run's Bronze batch in memory for the next Silver step
    if bronze_batch_df is None:
        bronze_batch_df = one_file_bronze_df
    else:
        bronze_batch_df = bronze_batch_df.unionByName(one_file_bronze_df, allowMissingColumns=True)

print("Bronze rows written this run:", 0 if bronze_batch_df is None else bronze_batch_df.count())

StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 83, Finished, Available, Finished, False)

Skipping already loaded Bronze file: Sales_2019_20260404_V1.csv
Skipping already loaded Bronze file: Sales_2020_2026_0404_V1.csv
Bronze rows written this run: 2735


**Build Silver updates from this Bronze batch**

In [82]:
if bronze_batch_df is None:
    print("No new Bronze files loaded in this run. Silver step skipped.")
else:
    silver_updates_df = (
        bronze_batch_df
        .withColumn("SalesOrderNumber", trim(col("SalesOrderNumber")))
        .withColumn("SalesOrderLineNumber", col("SalesOrderLineNumber").cast("int"))
        .withColumn(
            "OrderDate",
            coalesce(
                to_date(trim(col("OrderDate")), "yyyy-MM-dd"),
                to_date(trim(col("OrderDate")), "M/d/yy"),
                to_date(trim(col("OrderDate")), "MM/dd/yy"),
                to_date(to_timestamp(trim(col("OrderDate")), "yyyy-MM-dd HH:mm:ss")),
                to_date(to_timestamp(trim(col("OrderDate")), "M/d/yy HH:mm:ss")),
                to_date(to_timestamp(trim(col("OrderDate")), "MM/dd/yy HH:mm:ss"))
            )
        )
        .withColumn("CustomerName", trim(col("CustomerName")))
        .withColumn("Email", lower(trim(col("Email"))))
        .withColumn("Item", trim(col("Item")))
        .withColumn("Quantity", col("Quantity").cast("int"))
        .withColumn("UnitPrice", col("UnitPrice").cast("double"))
        .withColumn("Tax", col("Tax").cast("double"))
        .withColumn(
            "SnapshotDate",
            coalesce(
                to_date(trim(col("SnapshotDate")), "yyyy-MM-dd"),
                to_date(trim(col("SnapshotDate")), "M/d/yy"),
                to_date(trim(col("SnapshotDate")), "MM/dd/yy"),
                to_date(to_timestamp(trim(col("SnapshotDate")), "yyyy-MM-dd HH:mm:ss")),
                to_date(to_timestamp(trim(col("SnapshotDate")), "M/d/yy HH:mm:ss")),
                to_date(to_timestamp(trim(col("SnapshotDate")), "MM/dd/yy HH:mm:ss"))
            )
        )
        .withColumn("IsFlagged", when(col("Quantity") <= 0, lit(True)).otherwise(lit(False)))
        .withColumn("FileName", col("SourceFileName"))
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedTS", current_timestamp())
        .withColumn("ModifiedTS", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .select(
            "SalesOrderNumber",
            "SalesOrderLineNumber",
            "OrderDate",
            "CustomerName",
            "Email",
            "Item",
            "Quantity",
            "UnitPrice",
            "Tax",
            "SnapshotDate",
            "IsFlagged",
            "FileName",
            "LoadTimestamp",
            "CreatedTS",
            "ModifiedTS",
            "CreatedBy",
            "ModifiedBy"
        )
        .filter(
            col("SalesOrderNumber").isNotNull() &
            col("SalesOrderLineNumber").isNotNull()
        )
    )

    print("Silver updates rows:", silver_updates_df.count())

StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 84, Finished, Available, Finished, False)

Silver updates rows: 2735


**Merge into Silver table at order-line grain**

In [83]:
if bronze_batch_df is None:
    print("No new Bronze batch found. Silver merge skipped.")
else:
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    silver_delta.alias("tgt").merge(
        silver_updates_df.alias("src"),
        """
        tgt.SalesOrderNumber = src.SalesOrderNumber
        AND tgt.SalesOrderLineNumber = src.SalesOrderLineNumber
        """
    ).whenMatchedUpdate(set={
        "OrderDate": "src.OrderDate",
        "CustomerName": "src.CustomerName",
        "Email": "src.Email",
        "Item": "src.Item",
        "Quantity": "src.Quantity",
        "UnitPrice": "src.UnitPrice",
        "Tax": "src.Tax",
        "SnapshotDate": "src.SnapshotDate",
        "IsFlagged": "src.IsFlagged",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "ModifiedTS": "src.ModifiedTS",
        "ModifiedBy": "src.ModifiedBy"
    }).whenNotMatchedInsert(values={
        "SalesOrderNumber": "src.SalesOrderNumber",
        "SalesOrderLineNumber": "src.SalesOrderLineNumber",
        "OrderDate": "src.OrderDate",
        "CustomerName": "src.CustomerName",
        "Email": "src.Email",
        "Item": "src.Item",
        "Quantity": "src.Quantity",
        "UnitPrice": "src.UnitPrice",
        "Tax": "src.Tax",
        "SnapshotDate": "src.SnapshotDate",
        "IsFlagged": "src.IsFlagged",
        "FileName": "src.FileName",
        "LoadTimestamp": "src.LoadTimestamp",
        "CreatedTS": "src.CreatedTS",
        "ModifiedTS": "src.ModifiedTS",
        "CreatedBy": "src.CreatedBy",
        "ModifiedBy": "src.ModifiedBy"
    }).execute()

    print("Silver merge completed.")

StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, 85, Finished, Available, Finished, False)

UnsupportedOperationException: [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same
target row in the Delta table in possibly conflicting ways. By SQL semantics of Merge,
when multiple source rows match on the same target row, the result may be ambiguous
as it is unclear which source row should be used to update or delete the matching
target row. You can preprocess the source table to eliminate the possibility of
multiple matches. Please refer to
https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge

**Validate Bronze and Silver output**

In [ ]:
print("Bronze count:", spark.read.table(BRONZE_TABLE).count())
print("Silver count:", spark.read.table(SILVER_TABLE).count())

print("Silver distinct order-line count:",
      spark.read.table(SILVER_TABLE)
           .select("SalesOrderNumber", "SalesOrderLineNumber")
           .distinct()
           .count())

display(spark.read.table(BRONZE_TABLE).orderBy(desc("IngestedAt")))
display(spark.read.table(SILVER_TABLE).orderBy(desc("LoadTimestamp")))


StatementMeta(, aff25f42-7b61-4519-8ac1-dba6915a1987, -1, Cancelled, , Cancelled, True)